# 🧪 معمل التجارب: إضافة ذاكرة المحادثة لمسار الـ RAG (Lecture 30)
دليل تطبيقي مباشر لبناء **Conversational RAG** باستخدام `create_history_aware_retriever` و `MessagesPlaceholder` وسجل الحوار `chat_history` حتى يستوعب النظام الأسئلة التتابعية ويفهم عَمَّن تعود الضمائر.

### 1. استيراد المكتبات وتحديد مسار قاعدة البيانات
**آلية العمل:**
نقوم بتحميل متغيرات البيئة واستيراد مكونات المحادثة من لانج تشين:
- `MessagesPlaceholder`: لحجز مكان ديناميكي لقائمة رسائل الحوار السابقة داخل الـ Prompt.
- `HumanMessage` و `AIMessage`: كائنات تمثل رسائل المستخدم والردود المولدة لتخزينها في الذاكرة.
- `create_history_aware_retriever`: دالة تدمج الـ LLM مع الـ Retriever لإعادة صياغة السؤال بسياق الحوار.
- `create_stuff_documents_chain` و `create_retrieval_chain`: لتجميع خط الأنابيب الكامل.

In [2]:
# import standard libraries and load environment variables
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# resolve centralized chroma database path
data_dir = Path("../../data") if Path("../../data").exists() else Path("./data")
db_dir = str(data_dir / "databases" / "chroma_vector_store")

# import langchain core components
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

# import chain constructors with compatibility fallback
try:
    from langchain.chains import create_history_aware_retriever, create_retrieval_chain
    from langchain.chains.combine_documents import create_stuff_documents_chain
except (ImportError, ModuleNotFoundError):
    from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain

print("All libraries imported successfully!")
print(f"Vector Store Path: {db_dir}")

All libraries imported successfully!
Vector Store Path: ..\..\data\databases\chroma_vector_store


### 2. الاتصال بقاعدة بيانات المتجهات وتجهيز المسترجع (Retriever)
**آلية العمل:**
نتصل بفهرس `ChromaDB` الموجود على القرص باستخدام نفس نموذج التضمين، ثم نحوله إلى `Retriever` يسترجع أفضل 3 قطع دلالية مطابقة (`k=3`).

In [3]:
# initialize embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# connect to persistent chroma vector store
vector_store = Chroma(
    persist_directory=db_dir,
    embedding_function=embeddings,
    collection_name="rag_collection"
)

# create base retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print(f"Connected to ChromaDB! Current vector count: {vector_store._collection.count()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Connected to ChromaDB! Current vector count: 18


### 3. إعداد نموذج اللغة (Groq LLM)
**آلية العمل:**
نقوم بتهيئة نموذج `ChatGroq` السريع مع ضبط درجة الحرارة على قيمة منخفضة (`temperature=0.2`) لضمان التزامه التام بالسياق وإعادة صياغة الأسئلة بدقة.

In [4]:
# initialize fast Groq LLM
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=512
)
print("Groq LLM initialized successfully!")

Groq LLM initialized successfully!


### 4. بناء المسترجع الواعي بسياق الحوار (`create_history_aware_retriever`)
**آلية العمل:**
هذا هو الابتكار الأساسي في المحاضرة 30:
1. ننشئ قالب توجيه يطلب من النموذج فحص سجل المحادثات (`chat_history`) والسؤال الجديد للمستخدم (`input`).
2. إذا كان السؤال يحتوي على ضمائر غامضة (مثل *"ما هي أنواعه؟"*)، يقوم النموذج بإعادة صياغته لسؤال مستقل كامل (مثل *"ما هي أنواع تعلم الآلة؟"*).
3. يتم إرسال السؤال المعاد صياغته فقط إلى قاعدة بيانات المتجهات للبحث الدلالي، مما يضمن استرجاع أدق النصوص المناسبة.

In [5]:
# system prompt instructing the LLM to reformulate follow-up queries
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as it is."
)

# prompt template incorporating chat history placeholder
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

# construct history-aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm=llm,
    retriever=retriever,
    prompt=contextualize_q_prompt
)
print("History-aware retriever created successfully!")

History-aware retriever created successfully!


### 5. بناء سلسلة الإجابة عن الأسئلة (`create_stuff_documents_chain`)
**آلية العمل:**
ننشئ قالب الإجابة النهائي الذي يستلم النصوص المسترجعة (`{context}`) مع سجل المحادثات (`chat_history`) وسؤال المستخدم، ليقوم النموذج بصياغة رد مباشر ودقيق ومستند فقط إلى السياق.

In [6]:
# system prompt for final question answering
qa_system_prompt = (
    "You are a helpful assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "Use three sentences maximum and keep the answer concise and well-structured.\n\n"
    "{context}"
)

# prompt template with system instruction, chat history, and user input
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

# combine documents chain
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
print("Question answering document chain created!")

Question answering document chain created!


### 6. دمج مسار الـ Conversational RAG بالكامل (`create_retrieval_chain`)
**آلية العمل:**
نربط `history_aware_retriever` مع `question_answer_chain` عبر دالة `create_retrieval_chain` لينتج خط أنابيب متكامل يدير تلقائياً:
1. إعادة صياغة السؤال.
2. البحث في المتجهات.
3. توليد الإجابة مع الحفاظ على الذاكرة.

In [7]:
# assemble full conversational retrieval chain
conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain
)
print("Conversational RAG chain is ready for multi-turn dialogue!")

Conversational RAG chain is ready for multi-turn dialogue!


### 7. الجولة الأولى من المحادثة (السؤال الافتتاحي)
**آلية العمل:**
نبدأ بسجل محادثة فارغ (`chat_history = []`) ونطرح السؤال الأول عن مفهوم عام: *"What is machine learning?"*.
بعد استلام الإجابة، نقوم بإلحاق كل من سؤال المستخدم كـ `HumanMessage` والرد المولد كـ `AIMessage` داخل قائمة `chat_history`.

In [8]:
# initialize empty conversation memory
chat_history = []

# turn 1: ask the first question
query_1 = "What is machine learning?"
response_1 = conversational_rag_chain.invoke({
    "input": query_1,
    "chat_history": chat_history
})

print(f"👤 Human: {query_1}\n")
print(f"🤖 Assistant:\n{response_1['answer']}\n")

# update chat history with turn 1 messages
chat_history.extend([
    HumanMessage(content=query_1),
    AIMessage(content=response_1["answer"])
])
print(f"Memory updated! Stored messages count: {len(chat_history)}")

👤 Human: What is machine learning?

🤖 Assistant:
Machine learning is a branch of artificial intelligence that enables computers to learn from data without explicit programming. It typically uses labeled datasets in supervised learning, applying algorithms such as regression, decision trees, and other learning models. These models can then accurately identify, classify, and react to objects, with key applications in image classification, object detection, and visual recognition.

Memory updated! Stored messages count: 2


### 8. الجولة الثانية من المحادثة (السؤال التتابعي الغامض)
**آلية العمل:**
هنا يكمن الاختبار الحقيقي للذاكرة:
نطرح سؤالاً ناقصاً لا يحتوي على اسم الموضوع صراحة: *"What are its main types?"* (ما هي أنواعه الرئيسية؟).
- بفضل `history_aware_retriever`، يفهم النظام أن كلمة `its` تعود على `machine learning` من تاريخ الجولة السابقة.
- يعيد صياغة الاستعلام في الخلفية ويبحث عن أنواع تعلم الآلة، ثم يجيب بدقة (Supervised, Unsupervised, Reinforcement)!

In [9]:
# turn 2: follow-up question referencing previous context via pronoun 'its'
query_2 = "What are its main types?"
response_2 = conversational_rag_chain.invoke({
    "input": query_2,
    "chat_history": chat_history
})

print(f"👤 Human: {query_2}\n")
print(f"🤖 Assistant (Aware of Context):\n{response_2['answer']}\n")

# update chat history with turn 2
chat_history.extend([
    HumanMessage(content=query_2),
    AIMessage(content=response_2["answer"])
])

print("=" * 60)
print(f"Total conversation history length: {len(chat_history)} messages")

👤 Human: What are its main types?

🤖 Assistant (Aware of Context):
The main types of machine learning are **supervised learning**, where models are trained on labeled data; **unsupervised learning**, which discovers patterns in unlabeled data; and **reinforcement learning**, where agents learn by receiving rewards or penalties from their environment. Each type uses different algorithms and is suited to distinct problem domains.

Total conversation history length: 4 messages


### 9. التحقق من المستندات المسترجعة (Retrieved Chunks)
**آلية العمل:**
قام القاموس المرجع `response_2` بتخزين القطع المسترجعة داخل مفتاح `context`.
نفحص هذه القطع لنتأكد بالدليل القاطع أن المسترجع بحث بالفعل عن نصوص `types of machine learning` وليس عن مجرد كلمات عامة.

In [10]:
# inspect context chunks used for the follow-up answer
print("📚 Context Chunks retrieved for the follow-up question:")
for idx, doc in enumerate(response_2.get("context", []), start=1):
    src = doc.metadata.get("source", "unknown")
    topic = doc.metadata.get("topic", "general")
    print(f"  [{idx}] (Source: {src} | Topic: {topic}):")
    print(f"      {doc.page_content[:90]}...")

📚 Context Chunks retrieved for the follow-up question:
  [1] (Source: ..\..\data\chroma_sample_data\document_0.txt | Topic: general):
      Machine learning is a subset of artificial intelligence that learns from data without expl...
  [2] (Source: ..\..\data\chroma_sample_data\doc_0.txt | Topic: general):
      Machine learning is a subset of artificial intelligence that learns from data without expl...
  [3] (Source: docs_3.txt | Topic: computer_vision):
      learning models, machines can accurately identify and classify objects, and then react to ...


## 🎯 ملخص الدرس 05 (المحاضرة 30):
1. **`create_history_aware_retriever`**: يحل مشكلة غياب السياق عن طريق إعادة صياغة السؤال التتابعي آلياً قبل البحث في الفهرس.
2. **`MessagesPlaceholder("chat_history")`**: يسمح بتمرير قائمة رسائل متغيرة ديناميكياً داخل قوالب المحادثة.
3. **`chat_history.extend([HumanMessage, AIMessage])`**: هي الآلية اليدوية والمرنة لتحديث الذاكرة بعد كل تبادل حواري.
4. **النتيجة**: تحول النظام من مجرد محرك بحث صامت إلى مساعد محادثة حقيقي وذكي (Conversational AI Assistant).